## Case Study Task
* 1. What is the market share of our company202 parts vs the competition?
* 2. Compare the defect frequency of our parts vs the competition for OEM 1 cars
* 3. How many Engines have parts from our company build in?
* 4. Create a Dashboard visualization

## 1.1 Writing vodoo helper function for importing dirty Einzelteile

In [133]:
import io
from pathlib import Path
import pandas as pd

DATA = Path("data/Einzelteil")

FILES = {
    "t01": ("Einzelteil_T01.txt", b" | | ", b" "),
    "t02": ("Einzelteil_T02.txt", b"  ", b"\t"),
    "t03": ("Einzelteil_T03.txt", b"|", b"\x0b"),
    "t04": ("Einzelteil_T04.csv", b";", b"\n"),
    "t05": ("Einzelteil_T05.csv", b",", b"\n"),
}

def load(key: str) -> pd.DataFrame:
    filename, field_sep, row_sep = FILES[key]
    
    # Byte vodoo (replacing field- and row- seperators)
    raw = (DATA / filename).read_bytes().replace(field_sep, b"\x01").replace(row_sep, b"\n")
    df = pd.read_csv(io.BytesIO(raw), sep="\x01", na_values=["NA"], dtype=str)

    # Combine same columns (.x, .y)
    for col in set(c.removesuffix(".x").removesuffix(".y") for c in df.columns):
        if col + ".x" in df.columns and col + ".y" in df.columns:
            df[col] = df[col + ".x"].combine_first(df[col + ".y"])

    # Standartize id and time columns
    part = key.upper()
    df["Part_ID"] = df.get(f"ID_{part}", df.get("Part_ID"))

    if "Produktionsdatum" not in df.columns and "Produktionsdatum_Origin_01011970" in df.columns:
        df["Produktionsdatum"] = pd.to_datetime("1970-01-01") + pd.to_timedelta(
            df["Produktionsdatum_Origin_01011970"].astype(float), unit="D"
        )

    # Convert datatypes
    df["Produktionsdatum"] = pd.to_datetime(df["Produktionsdatum"])
    df["Fehlerhaft_Datum"] = pd.to_datetime(df["Fehlerhaft_Datum"])
    df["Fehlerhaft_Fahrleistung"] = df["Fehlerhaft_Fahrleistung"].str.replace(",", ".", regex=False).astype(float)
    
    int_cols = ["Herstellernummer", "Werksnummer", "Fehlerhaft"]
    df[int_cols] = df[int_cols].astype("Int64")

    columns = [
        "Part_ID", 
        "Herstellernummer", 
        "Werksnummer", 
        "Produktionsdatum", 
        "Fehlerhaft", 
        "Fehlerhaft_Datum", 
        "Fehlerhaft_Fahrleistung"
    ]

    return df[columns].reset_index(drop=True)

## 1.2 Importing relevant Einzelteile Files



In [134]:
# Loading the part files by using our helper function
t01 = load("t01")
t02 = load("t02")
t03 = load("t03")
t04 = load("t04")
t05 = load("t05")

einzelteile = [t01, t02, t03, t04, t05]
display(t01.head(), t02.head(), t03.head(), t04.head(), t05.head())

,Part_ID,Herstellernummer,Werksnummer,Produktionsdatum,Fehlerhaft,Fehlerhaft_Datum,Fehlerhaft_Fahrleistung
0,1-201-2011-247,201,2011,2008-11-07,0,NaT,0.0
1,1-201-2011-429,201,2011,2008-11-07,0,NaT,0.0
2,1-201-2011-363,201,2011,2008-11-07,1,2009-09-30,12983.0
3,1-201-2011-30,201,2011,2008-11-07,0,NaT,0.0
4,1-201-2011-72,201,2011,2008-11-07,1,2009-09-30,12983.0


,Part_ID,Herstellernummer,Werksnummer,Produktionsdatum,Fehlerhaft,Fehlerhaft_Datum,Fehlerhaft_Fahrleistung
0,2-201-2011-239,201,2011,2008-11-07,1,2010-04-09,38354.158904
1,2-201-2011-304,201,2011,2008-11-07,0,NaT,0.000000
2,2-201-2011-125,201,2011,2008-11-07,1,2010-04-09,38354.158904
3,2-201-2011-55,201,2011,2008-11-07,0,NaT,0.000000
4,2-201-2011-133,201,2011,2008-11-07,0,NaT,0.000000


,Part_ID,Herstellernummer,Werksnummer,Produktionsdatum,Fehlerhaft,Fehlerhaft_Datum,Fehlerhaft_Fahrleistung
0,3-202-2023-249,202,2023,2008-11-07,0,NaT,0.0
1,3-202-2022-8,202,2022,2008-11-07,0,NaT,0.0
2,3-202-2023-192,202,2023,2008-11-07,0,NaT,0.0
3,3-202-2023-16,202,2023,2008-11-07,0,NaT,0.0
4,3-202-2023-258,202,2023,2008-11-07,0,NaT,0.0


,Part_ID,Herstellernummer,Werksnummer,Produktionsdatum,Fehlerhaft,Fehlerhaft_Datum,Fehlerhaft_Fahrleistung
0,4-204-2043-113,204,2043,2008-11-07,0,NaT,0.0
1,4-202-2023-18,202,2023,2008-11-07,0,NaT,0.0
2,4-204-2043-98,204,2043,2008-11-07,0,NaT,0.0
3,4-202-2023-51,202,2023,2008-11-07,0,NaT,0.0
4,4-204-2043-169,204,2043,2008-11-07,0,NaT,0.0


,Part_ID,Herstellernummer,Werksnummer,Produktionsdatum,Fehlerhaft,Fehlerhaft_Datum,Fehlerhaft_Fahrleistung
0,5-201-2012-82,201,2012,2008-11-07,1,2010-07-10,43163.430137
1,5-201-2012-172,201,2012,2008-11-07,0,NaT,0.000000
2,5-201-2012-23,201,2012,2008-11-07,0,NaT,0.000000
3,5-201-2012-47,201,2012,2008-11-07,1,2010-07-10,43163.430137
4,5-201-2012-101,201,2012,2008-11-07,0,NaT,0.000000


## 1.3 Importing component and OEM1 files

In [135]:
file_paths = [
    'data/Komponente/Bestandteile_Komponente_K1BE1.csv',
    'data/Komponente/Bestandteile_Komponente_K1DI1.csv',
    'data/Komponente/Bestandteile_Komponente_K1BE2.csv',
    'data/Komponente/Bestandteile_Komponente_K1DI2.csv'
]

engine_dfs = []

for path in file_paths:
    df = pd.read_csv(path, sep=';').drop(columns=['Unnamed: 0'])
    engine_dfs.append(df)


# Importing OEM1 data
oem11 = pd.read_csv("data/Fahrzeug/Bestandteile_Fahrzeuge_OEM1_Typ11.csv", sep=";").drop(columns=["Unnamed: 0"], errors="ignore")
oem12 = pd.read_csv("data/Fahrzeug/Bestandteile_Fahrzeuge_OEM1_Typ12.csv", sep=";").drop(columns=["Unnamed: 0"], errors="ignore")

# Combining the OEM1 Types
oem_combined = pd.concat([oem11, oem12], ignore_index=True)

## 1.4 Add OEM1 flag to the einzelteile dataframes

In [136]:
# Get motor_ids from OEM1
oem_motor_ids = oem_combined["ID_Motor"]

# Check if component ids contain the oem1 motor ids
for df in engine_dfs:
    id_col = df.columns[4]
    df["In_OEM1"] = df[id_col].isin(oem_motor_ids).astype(int)

# Extract part IDs from OEM1-flagged engines
oem1_part_ids = set()

for df in engine_dfs:
    oem1_components = df[df['In_OEM1'] == 1]
    part_cols = [col for col in df.columns if col.startswith('ID_T')]
    for col in part_cols:
        oem1_part_ids.update(oem1_components[col].dropna())

# Check if part ids are in OEM1
for t_df in einzelteile:
    t_df['In_OEM1'] = t_df['Part_ID'].isin(oem1_part_ids).astype(int)

display(engine_dfs[0])

,ID_T1,ID_T2,ID_T3,ID_T4,ID_K1BE1,In_OEM1
0,1-201-2011-45,2-201-2011-161,3-202-2023-14,4-202-2023-20,K1BE1-101-1011-1,1
1,1-201-2011-429,2-201-2011-239,3-202-2023-16,4-202-2023-51,K1BE1-101-1011-2,1
2,1-201-2011-399,2-201-2011-220,3-202-2023-46,4-202-2023-93,K1BE1-101-1011-3,1
3,1-201-2011-335,2-202-2022-463,3-202-2023-149,4-204-2042-18,K1BE1-101-1011-4,1
4,1-204-2044-188,2-202-2022-675,3-202-2023-152,4-204-2042-40,K1BE1-101-1011-5,1
...,...,...,...,...,...,...
1192625,1-201-2011-1278204,2-202-2022-2239326,3-202-2023-855660,4-204-2042-140172,K1BE1-104-1041-712653,1
1192626,1-202-2021-1278715,2-202-2022-2239792,3-202-2023-855668,4-204-2042-140181,K1BE1-104-1041-712654,1
1192627,1-202-2021-1278249,2-202-2022-2239781,3-202-2023-855673,4-204-2042-140192,K1BE1-104-1041-712655,1
1192628,1-202-2021-1278549,2-202-2022-2239824,3-202-2023-855544,4-204-2042-140165,K1BE1-104-1041-712656,1


## 1.3 Collecting Data and creating a comprehensive dataframe for market share

In [137]:
import pandas as pd

def get_einzelteile_stats(data_list, oem1_only=False):
    # Optionaler filter for OEM1
    if oem1_only:
        data_list = [df[df['In_OEM1'] == 1] for df in data_list]

    stats = []
    names = ['T01', 'T02', 'T03', 'T04', 'T05']

    for name, df in zip(names, data_list):
        total_ids = df['Part_ID'].nunique()
        
        # Filter by manufacturer (202 vs. competition)
        df_202 = df[df['Herstellernummer'] == 202]
        df_comp = df[df['Herstellernummer'] != 202]
        
        ids_202 = df_202['Part_ID'].nunique()
        ids_comp = total_ids - ids_202
        
        # Count faulty parts for both manufacturer 202 and competition
        faulty_202 = df_202[df_202['Fehlerhaft'] == 1]['Part_ID'].nunique()
        faulty_comp = df_comp[df_comp['Fehlerhaft'] == 1]['Part_ID'].nunique()
        
        # Calculate shares and failure rates
        share_202 = round((ids_202 / total_ids * 100), 2)
        fail_rate_202 = round((faulty_202 / ids_202 * 100), 2)
        fail_rate_comp = round((faulty_comp / ids_comp * 100), 2)

        stats.append({
            'einzelteile': name,
            'total_unique_part_id': total_ids,
            'unique_part_id_202': ids_202,
            'unique_part_id_competition': ids_comp,
            'relative_part_id_202_%': share_202,
            'faulty_unique_part_id_202': faulty_202,
            'faulty_unique_part_id_competition': faulty_comp,
            'relative_faulty_unique_part_id_202_%': fail_rate_202,
            'relative_faulty_unique_part_id_competition_%': fail_rate_comp
        })

    count_df = pd.DataFrame(stats)
    
    # Calculate overall statistics
    total_parts = count_df['total_unique_part_id'].sum()
    parts_202 = count_df['unique_part_id_202'].sum()
    
    rel_total = (parts_202 / total_parts * 100) if total_parts > 0 else 0.0
    
    dataset_label = "OEM1" if oem1_only else "Total"
    print(f"[{dataset_label}] {round(rel_total, 2)}% of all parts came from manufacturer 202.")
    print(f"[{dataset_label}] manufacturer 202 has delivered a total of {parts_202} parts.")
    
    return count_df

In [138]:
# Für den Gesamtdatensatz:
einzelteile_count_all = get_einzelteile_stats(einzelteile, oem1_only=True)
display(einzelteile_count_all)

[OEM1] 59.09% of all parts came from manufacturer 202.
[OEM1] manufacturer 202 has delivered a total of 4651291 parts.


,einzelteile,total_unique_part_id,unique_part_id_202,unique_part_id_competition,relative_part_id_202_%,faulty_unique_part_id_202,faulty_unique_part_id_competition,relative_faulty_unique_part_id_202_%,relative_faulty_unique_part_id_competition_%
0,T01,1908869,954029,954840,49.98,190610,238183,19.98,24.94
1,T02,2385260,1669791,715469,70.00,166642,71451,9.98,9.99
2,T03,1192630,1073367,119263,90.00,107305,11800,10.00,9.89
3,T04,1192630,357789,834841,30.00,35503,84112,9.92,10.08
4,T05,1192630,596315,596315,50.00,59392,59412,9.96,9.96


## 3.3 Filtering dataset using regex to avoid complicated merges

In [139]:
# Creating a function to filter by 202 company
def filter_mask(df):
    mask = False
    columns = df.columns[0:4]
    for col in columns:
        mask = mask | df[col].str.contains(r'-202-', na=False)
    return mask

# Creating empty List
filtered_engine_dfs = []

# Creating bool mask and applying the mask
for df in engine_dfs:
    mask = filter_mask(df)
    filtered_engine_dfs.append(df[mask])


## 3.4 Calculating how many engines use our company202 parts

In [140]:
# Part use rates
percentages = []

for i in range(len(engine_dfs)):
    percentages.append(round(len(filtered_engine_dfs[i]) / len(engine_dfs[i]) * 100, 2))


# Converting use rates into our advertising slogan
engine_names = ['K1BE1', 'K1DI1', 'K1BE2', 'K1DI2']

for i in range(len(engine_names)):
    ratio = round(100 / percentages[i], 2)
    print(f"every {ratio} th {engine_names[i]} engine contains parts from 202 manufacturer")

# Calculating sums from the lists
summe_aller_engines_mit_202 = sum(len(df) for df in filtered_engine_dfs)
summe_aller_engines = sum(len(df) for df in engine_dfs)

# Calculating percentages and ratios
percentage_aller_engines_mit_202 = round(summe_aller_engines_mit_202 / summe_aller_engines * 100, 2)
ratio_all = round(100 / percentage_aller_engines_mit_202, 2)

# printing total slogan
print(f"In every {ratio_all}th engine there are parts from 202.")
print(f"{percentage_aller_engines_mit_202}% of all engines contain parts from our 202 company")


every 1.01 th K1BE1 engine contains parts from 202 manufacturer
every 1.1 th K1DI1 engine contains parts from 202 manufacturer
every 1.22 th K1BE2 engine contains parts from 202 manufacturer
every 1.22 th K1DI2 engine contains parts from 202 manufacturer
In every 1.09th engine there are parts from 202.
91.58% of all engines contain parts from our 202 company
